# Week 10: Frameworks and a Simple Chatbot

**Note:** I don't have your actual Week 3 chatbot file, so this notebook rebuilds a simple
"ask a question, get an answer" chatbot from scratch first (the "raw API" version), then
rebuilds the exact same thing using a framework. If your real Week 3 chatbot was different,
just swap the raw-API section below for your own code — the framework section still applies.

**What is a "framework" here?**

When you call `client.models.generate_content(...)` directly, that's the **raw API** —
you build everything yourself, including things like "remembering earlier messages".

A **framework** (we're using **LangChain**) gives you ready-made building blocks for common
chatbot needs — like memory — so you don't have to write that logic by hand every time.

**What we'll do:**
1. Build a simple chatbot with the raw API (no memory — this is our "Week 3 style" baseline).
2. Rebuild it using LangChain.
3. Add one feature using the framework: **memory**, so the chatbot remembers earlier messages.
4. Compare the two approaches.

In [13]:
!pip install google-genai langchain langchain-google-genai python-dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise SystemExit("No GEMINI_API_KEY found.")

print("API key loaded.")

API key loaded.


## Part 1: The raw API chatbot (no framework)

This is a plain chatbot with nothing remembered from
before. This is our stand-in for a typical "Week 3 style" chatbot.

In [15]:
from google import genai

client = genai.Client(api_key=api_key)

def raw_chat(question):
    """Sends one question straight to Gemini. It has no memory of past questions."""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=question
    )
    return response.text

print("Q1:", "My name is Sam.")
print("A1:", raw_chat("My name is Sam."))

print("\nQ2:", "What is my name?")
print("A2:", raw_chat("What is my name?"))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Q1: My name is Sam.
A1: Hello Sam! Nice to meet you. How can I help you today?

Q2: What is my name?
A2: I don't know your name yet! Since I don't have access to your personal information, you'll have to tell me. What is your name?


## Part 2: The same chatbot, rebuilt with LangChain

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI

chat_model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=api_key
)

# .invoke() sends one message and gets one reply back, same idea as raw_chat above.
response = chat_model.invoke("What is a chatbot, in one short sentence?")
print(response.content)

[{'type': 'text', 'text': 'A chatbot is a software application designed to simulate human conversation through text or voice interactions.', 'extras': {'signature': 'EsELCr4LARFNMg+1u6F/mIegrrIFYXoA/gnxovIqHO02Mm5k+S4jg7r0DfxBuyqQkmqI6z7Izv5mauxanSzmMQy96Y5n4CKqp7rSWl1aqyIzPxV/ayHP/RQpWoHRoTKMdNVBRH60NyxSUILW41NSzi7hywBRD/DbVVwjLbajSpbZk6V05/MJQovMZ0fF7itZllgM6RseXiv+rB4ZMLlj6LAwvbZnQ/Tsr/K4//okZ7viLi6qD9+9u5XnoFp6j022jGrQ1yeaPcdi38bb/TGKaqZcJmixnLueDEEwtWDQOTHMZVzZd22GZ0GRMdgE32lGaaqtLzW77zyZB+y9HNOjuFasWH3PdufLr2aY/YocfNzthKI0qi78/u1GBVAUI5SAu1DQf0twgTVDdrO3Awg9/zK/ZxoJ9/zDIka5r18nFv4TXc+4uPIMNAkxn9aWao6bOljcgOcgkS+GTvhMLDY8epOSupBCBIonoKw0hw7UzV8wIy97agnAS+ndXeYPAzux8tyR8zaPwwW1EJd5X6BRiMGOj9/CQUpL+6rMmVquYB2wIXjkZTzT6G+qGDiDRocLSsXCSdlS/IwEtV18pn2LP/ne86vCwd9PtaA9CSFYP4gWeTzIPTIs52k+WUwrOezk88UqPiPOtcRfU1NPkA6bBAXurf9p+bofSmzZ2DQzsDUOQelDDtOo0ox3h5uJSRIXU5V7vZV2bSltChscke7vH8IPJtY98WPu6YXr9aqvUzVMhIGxv449uRByr+a6cYZzkeMqu3lI+mdTlgO0tn1JoRSsfzg9DYjaREIbAfkyZKmcF0Wyrn4q+oe/ylhmnZjpx2

## Part 3: Add a feature: memory

This is the "one added feature" the Week 10 task asks for. `ConversationBufferMemory` keeps
track of everything said so far, and `ConversationChain` automatically includes that history
every time it talks to Gemini so the chatbot can actually remember earlier messages.

In [25]:
from langchain_core.messages import HumanMessage, AIMessage

# This list IS the memory. Every message, ours and the bot's, gets added to it.
chat_history = []

def chat_with_memory(user_input):
    """Sends the full conversation so far to Gemini, then remembers the new reply."""
    chat_history.append(HumanMessage(content=user_input))

    # We pass the WHOLE history, not just the latest message, so the model can
    # see everything that was said before.
    response = chat_model.invoke(chat_history)

    chat_history.append(AIMessage(content=response.content))
    return response.content


print("Q1:", "My name is Aastha.")
print("A1:", chat_with_memory("My name is Aastha."))

print("\nQ2:", "What is my name?")
print("A2:", chat_with_memory("What is my name?"))

Q1: My name is Aastha.
A1: [{'type': 'text', 'text': "Hello Aastha! It's lovely to meet you. How can I help you today?", 'extras': {'signature': 'EooFCocFARFNMg9uakDJY7BzA/tH0iN+9Ohfabi0SCAh0+zKzfKW2XA9uuOw0+GKkaQ9RWvoFjITPzdbToXfUVbEb+VBJwP8NJg7JnvlCl3aMAZTZtOBZ0cw0mqjcqPeXutOej5eBWohaPBYRnoZy5IN9blDxghd4NMSmpyc0MMxqWovykeAq44Q0epmgBx9Ocbco1e9cZCCAwsSbSJtV8p7aPjs5ip5Nf0ES1fNbLLcTa2nmKazh8YoougeFNew4uPDm7zKhSQFV/ZaAqNMBfkbROZ/asTxgPFphnTVVLccEQb15aSZDvdlejhV1+bMvh258+yg0tbZTKhmMTVQdYLx4FTAW5pA+yoeQ6gDsYFEoIvpHRzAuubgRovxyzfakqyVzpuIo4EveTR8i6eZtOJPjbPAnm4yX5pw47GL7tx3Ted63GkxigbR8JONGEETQq/vdr+55JbUuf/1++3YjO5yuxdkprkVszydRvqlIUIapjhlgHeIu13VO6GcgvBuJaw9DUaszt0E5PDQDttBTxdL0GjDRz9v+HrpcmKYAAqxhKOHown75/I76MmAakEirKyXEpR4UKfpJSGZAYoaqm1Rl1q/Qn2WVTazu7F/9/+aqiU4z8YYHRn/vjHYhTMnXldy7iIhMHnv66Z2nlOOnON4HKWZz3AZB3l1j8Hneavv80et96tai25AxyOQYKlf1jkS/gTZpbY1MsxvxVexLpJagQ/+UyIgEx5xiha39Ppvwd/8V4H+3rAGCIQMSI+GrNf5FF/u6E6boFhC0bZwT7Zx4j+/Lmz/XFWYI55aIjwrSUQP/vbqeqJVK4/gmFvFPpP6BSxTmkyCauj+8c6tlE

## Part 4: A simple chat loop using the framework version

In [24]:
def run_chatbot():
    print("Framework chatbot with memory. Type 'exit' to stop.\n")

    while True:
        user_input = input("You: ").strip()

        if user_input == "":
            print("Please type something, or 'exit' to stop.\n")
            continue

        if user_input.lower() in ("exit", "quit"):
            print("Goodbye!")
            break

        try:
            reply = chat_with_memory(user_input)
            print("Bot:", reply, "\n")
        except Exception as error:
            print(f"Something went wrong: {error}\n")
